# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

#### Rule:
Rank by cluster archetype first, then by w04 score within each archetype:-
- Cluster -> Tells you what kind of problem an item has.
- Score -> Tells you how urgent it is within that kind.


#### Archetype fix:
My w05 naming rule collapsed all 4 clusters into HIGH_VOLUME_MIXED (flagged in w06). Replaced with relative-rank naming across my own 4 clusters instead of absolute cutoffs, with an assert that fails if it collapses again.

#### Archetypes:

- **Already Converting** — best CTR (2.08%), ~no gap → Monitor only

- **High-Reach Opportunity** — most impressions, most score → Fix title/meta, prioritized

- **Steady / Low Signal** — modest on every axis → Low-priority review

- **Low-Visibility / Buried** — worst position (45.8) → Flag for review, never auto-act

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv('/content/drive/MyDrive/baseline_action_score.csv')
print(f"Rows: {len(df):,}")

df['impressions_log'] = np.log10(df['impressions'] + 1)
features = ['ctr', 'avg_position', 'impressions_log']
for col in features:
    df[col] = df[col].fillna(df[col].median())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features].values)
km = KMeans(n_clusters=4, random_state=42, n_init=10, max_iter=300)
df['cluster'] = km.fit_predict(X_scaled)

print(df['cluster'].value_counts().sort_index())

cluster_profile = df.groupby('cluster').agg(
    n=('content_hash_id', 'count'),
    mean_ctr=('ctr', 'mean'),
    mean_position=('avg_position', 'mean'),
    mean_impressions=('impressions', 'mean'),
    mean_score=('score', 'mean'),
    pct_high_priority=('action_label', lambda s: 100 * (s == 'FIX_TITLE_META_HIGH_PRIORITY').mean()),
).reset_index()

cluster_profile['ctr_rank'] = cluster_profile['mean_ctr'].rank(ascending=False)
cluster_profile['position_rank'] = cluster_profile['mean_position'].rank(ascending=True)
cluster_profile['impressions_rank'] = cluster_profile['mean_impressions'].rank(ascending=False)
cluster_profile['score_rank'] = cluster_profile['mean_score'].rank(ascending=False)

print(cluster_profile.to_string(index=False))

def name_archetype(row):
    if row['ctr_rank'] == 1 and row['score_rank'] == cluster_profile['score_rank'].max():
        return 'Already Converting'
    if row['impressions_rank'] == 1 and row['score_rank'] == 1:
        return 'High-Reach Opportunity'
    if row['position_rank'] == cluster_profile['position_rank'].max():
        return 'Low-Visibility / Buried'
    return 'Steady / Low Signal'

cluster_profile['archetype'] = cluster_profile.apply(name_archetype, axis=1)

n_distinct = cluster_profile['archetype'].nunique()
print(f"Distinct archetype names: {n_distinct} / {len(cluster_profile)}")
assert n_distinct == len(cluster_profile), "Naming rule collapsed two clusters onto one label."

print(cluster_profile[['cluster', 'archetype', 'n', 'mean_ctr', 'mean_position', 'mean_impressions', 'mean_score']].to_string(index=False))

action_lookup = {
    'Already Converting':      ('MONITOR_ONLY',              'RC_HIGH_CTR_NO_GAP'),
    'High-Reach Opportunity':  ('FIX_TITLE_META_PRIORITIZED', 'RC_HIGH_REACH_CONFIRMED_GAP'),
    'Steady / Low Signal':     ('LOW_PRIORITY_REVIEW',        'RC_MODEST_ALL_AXES'),
    'Low-Visibility / Buried': ('REVIEW_FOR_PRUNE_OR_REWORK', 'RC_WORST_POSITION_LOW_SIGNAL'),
}
archetype_of = dict(zip(cluster_profile['cluster'], cluster_profile['archetype']))
df['archetype'] = df['cluster'].map(archetype_of)
df['playbook_action'] = df['archetype'].map(lambda a: action_lookup[a][0])
df['playbook_reason_code'] = df['archetype'].map(lambda a: action_lookup[a][1])

df['priority_rank'] = df.groupby('archetype')['score'].rank(ascending=False, method='first')

ranked_queue = df.sort_values(['archetype', 'priority_rank'])[
    ['content_hash_id', 'client_hash_id', 'archetype', 'playbook_action', 'playbook_reason_code',
     'action_label', 'reason_code', 'score', 'priority_rank', 'ctr', 'avg_position', 'impressions']
]
ranked_queue.head(15)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: 182,135
cluster
0    77898
1    29210
2    67701
3     7326
Name: count, dtype: int64
 cluster     n  mean_ctr  mean_position  mean_impressions  mean_score  pct_high_priority  ctr_rank  position_rank  impressions_rank  score_rank
       0 77898  0.002352      13.734875        808.006611    1.281583           0.000000       3.0            3.0               4.0         3.0
       1 29210  0.000915      45.787531       1458.572886    1.607351           0.010270       4.0            4.0               3.0         2.0
       2 67701  0.003170      11.763558      24084.845985   32.983685           7.781274       2.0            2.0               1.0         1.0
       3  7326  0.020920      10.250906       3355.349031    0.000000           0.000000       1.0            1.0               2.0         4.0
Distinct archetype names: 4 / 4
 cluster               arch

,content_hash_id,client_hash_id,archetype,playbook_action,playbook_reason_code,action_label,reason_code,score,priority_rank,ctr,avg_position,impressions
125590,content_771593dbcf6ad827,client_3ffa76342f366962,Already Converting,MONITOR_ONLY,RC_HIGH_CTR_NO_GAP,NO_ACTION_CTR_ON_TRACK,CTR_BELOW_EXPECTED_FOR_POSITION,0.0,1.0,0.022505,5.925795,1022.0
125604,content_7fb7671615874666,client_2094c6eb080311d5,Already Converting,MONITOR_ONLY,RC_HIGH_CTR_NO_GAP,NO_ACTION_CTR_ON_TRACK,CTR_BELOW_EXPECTED_FOR_POSITION,0.0,2.0,0.015267,6.885548,131.0
125611,content_5fbdd685e44e566c,client_f623b01661d4bfe4,Already Converting,MONITOR_ONLY,RC_HIGH_CTR_NO_GAP,NO_ACTION_CTR_ON_TRACK,CTR_BELOW_EXPECTED_FOR_POSITION,0.0,3.0,0.014144,23.794290,1414.0
125615,content_4c04346822b94ad7,client_20259bd6705d81d4,Already Converting,MONITOR_ONLY,RC_HIGH_CTR_NO_GAP,NO_ACTION_CTR_ON_TRACK,CTR_BELOW_EXPECTED_FOR_POSITION,0.0,4.0,0.019202,5.691223,3958.0
125622,content_d613438b52035da0,client_20259bd6705d81d4,Already Converting,MONITOR_ONLY,RC_HIGH_CTR_NO_GAP,NO_ACTION_CTR_ON_TRACK,CTR_BELOW_EXPECTED_FOR_POSITION,0.0,5.0,0.015913,4.464769,10683.0
125625,content_8a299c668f180d69,client_2094c6eb080311d5,Already Converting,MONITOR_ONLY,RC_HIGH_CTR_NO_GAP,NO_ACTION_CTR_ON_TRACK,CTR_BELOW_EXPECTED_FOR_POSITION,0.0,6.0,0.012567,11.255023,557.0
125628,content_c99d796c3ee5318c,client_23a62021009f63c4,Already Converting,MONITOR_ONLY,RC_HIGH_CTR_NO_GAP,NO_ACTION_CTR_ON_TRACK,CTR_BELOW_EXPECTED_FOR_POSITION,0.0,7.0,0.012712,14.422874,944.0
125630,content_36fab41f31ba7825,client_e5c2aa26a8598242,Already Converting,MONITOR_ONLY,RC_HIGH_CTR_NO_GAP,NO_ACTION_CTR_ON_TRACK,CTR_BELOW_EXPECTED_FOR_POSITION,0.0,8.0,0.032142,10.662067,40757.0
125631,content_d7b8bed1957ce6fc,client_cd12bcfd98942aa1,Already Converting,MONITOR_ONLY,RC_HIGH_CTR_NO_GAP,NO_ACTION_CTR_ON_TRACK,CTR_BELOW_EXPECTED_FOR_POSITION,0.0,9.0,0.022989,7.517642,174.0
125643,content_75c52066fce907dd,client_3ffa76342f366962,Already Converting,MONITOR_ONLY,RC_HIGH_CTR_NO_GAP,NO_ACTION_CTR_ON_TRACK,CTR_BELOW_EXPECTED_FOR_POSITION,0.0,10.0,0.028747,4.915549,487.0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

#### Intended use:

Internal decision-support to prioritize human review of ~182K content items into a ranked, archetype-grouped queue. Not for automated content changes.

#### Limits:

- **Shared-input risk:** score and cluster features share the same three raw signals (ctr, avg_position, impressions) — score correlates +0.70 with impressions alone. Archetype and score agreeing is a consistency check, but does not independently confirm that clustering found something new.

- **Descriptive only:** no semantic/topic signal, so the model can't judge content quality.

- **Generalization is real but modest:** the honest grouped-by-client silhouette is 0.335 on held-out clients, down from 0.373 on training clients.

- **One-month snapshot:** no trend signal. An "Already Converting" item today can be declining.

- **Archetype names:** relative-rank across these 4 clusters only re-derive on any refit with a different k or feature set, don't assume they carry over.

#### Where this stops being valid:

- After a major SERP/algorithm change; the w04 expected-CTR baseline reflects the current snapshot only.

- On a client or content type unlike the 59-client training mix.

- If corr(score, impressions) drifts far from +0.70; the archetype/score reasoning above assumes that holds.

- More than one refresh cycle since the last re-score; treat older labels as stale.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

#### The Rule:
Anything that changes or removes live content needs a person to approve it first. The model can rank items, but it can't make the final call.

#### Needs a human to check first:

- Anything marked REVIEW_FOR_PRUNE_OR_REWORK, no matter the score.

- Items with a negative silhouette score -> meaning the model itself is unsure which cluster they belong to, that also have a real score attached. w06 found 2,096 of these (1.2% of items) which is the biggest one and had a score of 282.8 and 237,716 impressions, and it only ended up in Low-Visibility because of how the cluster split fell. That's exactly the kind of case this rule catches.

- Any single action that would affect more than 5% of one client's total traffic.

- Any month where the archetype-naming check from Section 1 fails, if two clusters get the same name again, don't use that month's labels.

#### Never automate:
- Deleting or hiding content just because the score/archetype says so. Getting this wrong on a Low-Visibility item means real traffic loss overnight.

- Rewriting content based on which cluster it's in. Clustering only looks at numbers like CTR and position but it doesn't understand what the content actually says.

- Treating a significant ANOVA result as proof clustering found something new. w06 showed most of that result comes from score and cluster sharing the same inputs, not a real discovery.

- Acting on items below the 100-impression cutoff from w04 — the signal is too noisy that low.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

#### Retrain triggers:

- Silhouette score (on a held-out client split) drops well below the w06 baseline of 0.335 -> this number is what tells you the clusters generalize to clients they weren't built on. If it falls, the archetypes are starting to just memorize old clients rather than describe real, repeatable behavior; this implies the recommendations are going stale even if nothing else looks different.

- The archetype-naming check from Section 1 fails, fewer than 4 distinct names come out of the 4 clusters -> this means the naming rule can no longer tell the clusters apart in words, even if the underlying math still separates them fine. The queue would still run, but the archetype labels attached to it would be silently wrong.

- The score-impressions correlation moves far from the w06 baseline of +0.70 -> the limits section explicitly relies on that number to explain why archetype and score agree. If the correlation shifts, that explanation may no longer be true, so the recommendations rest on reasoning that no longer holds, not just on stale numbers.

- A w03 data-contract check breaks (row count, missingness pattern, grain) on new data -> this catches staleness at the source — if the input data itself changed shape, everything built on top of it is suspect before you even look at model output.


Re-check monthly, as each new month of data lands.

If the model hasn't been retrained or re-checked in 90+ days, treat the archetypes as stale by default and fall back to the plain w04 score-based ranking only — don't wait for one of the above to fire first.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

What gets exported, and why each one is handled differently:

- **work/outputs/w07_action_playbook_queue.csv** —> the full ranked queue: every item, its archetype, its action, its reason code, its rank. This is not committed to git — the CI leak-guard blocks data files, and this notebook regenerates the file fresh each run, so there's no reason to store it.

- **work/outputs/w07_playbook_summary.json** —> the numbers and rules behind the queue: cluster profile stats, archetype counts, action counts, the intended-use statement, the limits list, the no-go list, human-review rules, and the monitoring plan. This is committed — it's the receipt the paper's claims trace back to, so that later, I can quote a number from the paper and point straight back to this file.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.